# B5 - VAE on MNIST (latent dim 16)
Encoder: Flatten -> Dense256 -> Dense128 -> [mu(16), log_var(16)] -> z = mu + sigma x eps

Decoder: Dense128 -> Dense256 -> Dense784 sigmoid -> Reshape(28,28)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
# 1. Load + normalize to [0, 1]
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
train_ds = tf.data.Dataset.from_tensor_slices(x_train).shuffle(10000).batch(128)
LATENT = 16

In [ ]:
# 2. Encoder, reparameterization, decoder
class Sampling(layers.Layer):
    def call(self, mu, log_var):
        eps = tf.random.normal(tf.shape(mu))
        return mu + tf.exp(0.5 * log_var) * eps

class VAE(keras.Model):
    def __init__(self):
        super().__init__()
        self.encoder = keras.Sequential([layers.Input((28, 28)), layers.Flatten(),
                                         layers.Dense(256, activation='relu'),
                                         layers.Dense(128, activation='relu')], name='encoder')
        self.mu_layer = layers.Dense(LATENT, name='mu')
        self.logvar_layer = layers.Dense(LATENT, name='log_var')
        self.sampling = Sampling()
        self.decoder = keras.Sequential([layers.Input((LATENT,)),
                                         layers.Dense(128, activation='relu'),
                                         layers.Dense(256, activation='relu'),
                                         layers.Dense(784, activation='sigmoid'),
                                         layers.Reshape((28, 28))], name='decoder')

    def call(self, x):
        h = self.encoder(x)
        mu, log_var = self.mu_layer(h), self.logvar_layer(h)
        z = self.sampling(mu, log_var)
        return self.decoder(z), mu, log_var

def vae_loss(x, x_hat, mu, log_var):
    x_hat = tf.clip_by_value(x_hat, 1e-7, 1 - 1e-7)
    recon = -tf.reduce_sum(x * tf.math.log(x_hat) + (1 - x) * tf.math.log(1 - x_hat), axis=[1, 2])
    kl = -0.5 * tf.reduce_sum(1 + log_var - tf.square(mu) - tf.exp(log_var), axis=1)
    return tf.reduce_mean(recon + kl), tf.reduce_mean(recon), tf.reduce_mean(kl)

vae = VAE()
vae(x_train[:1])
vae.encoder.summary(); vae.decoder.summary()

In [ ]:
# 3. Train with reconstruction + KL loss
opt = keras.optimizers.Adam(1e-3)

@tf.function
def train_step(x):
    with tf.GradientTape() as tape:
        x_hat, mu, log_var = vae(x)
        loss, recon, kl = vae_loss(x, x_hat, mu, log_var)
    grads = tape.gradient(loss, vae.trainable_variables)
    opt.apply_gradients(zip(grads, vae.trainable_variables))
    return loss, recon, kl

hist = []
for epoch in range(10):
    loss, recon, kl = np.mean(np.array([train_step(x) for x in train_ds]), axis=0)
    hist.append(loss)
    print(f"Epoch {epoch+1}: loss={loss:.2f} recon={recon:.2f} KL={kl:.2f}")
plt.plot(hist); plt.xlabel('epoch'); plt.ylabel('loss'); plt.show()

In [ ]:
# 4. Test loss, reconstructions and generated digits
x_hat, mu, log_var = vae(x_test)
print("Test loss: %.2f (recon %.2f, KL %.2f)" % tuple(float(v) for v in vae_loss(x_test, x_hat, mu, log_var)))

plt.figure(figsize=(12, 3))
for i in range(10):
    plt.subplot(2, 10, i + 1); plt.imshow(x_test[i], cmap='gray'); plt.axis('off')
    plt.subplot(2, 10, 10 + i + 1); plt.imshow(x_hat[i], cmap='gray'); plt.axis('off')
plt.suptitle('Top: original   Bottom: reconstructed'); plt.show()

gen = vae.decoder(tf.random.normal((16, LATENT))).numpy()
plt.figure(figsize=(4, 4))
for i in range(16):
    plt.subplot(4, 4, i + 1); plt.imshow(gen[i], cmap='gray'); plt.axis('off')
plt.suptitle('Generated digits'); plt.show()